In [1]:
import numpy as np
import pandas as pd
import io
import scipy as sp
from scipy.stats import qmc
from numpy.random import Generator, PCG64

# Individual TE data

In [20]:
folder = '../Data/'
# df_AM22 = pd.read_excel(folder+"Armstrong McKay 2022/science.abn7950_data_s1.xlsx",
#                         skiprows=[0,1,2,3,4],sheet_name='4) Estimates',
#                         skipfooter=40,index_col=1,header=[0,1])
df_TEs = pd.read_excel(folder+'TEsummary.xlsx',header=[0],index_col=0)
df_TEs.head()

,Tipping element,Collapse (years),Recovery (years),Threshold (°C) – Min.,Threshold (°C) – Est.,Threshold (°C) – Max.,Beta (1/°C),Max Impact (°C) – Global,Comments
TE,,,,,,,,,
AMAZ,Amazon rainforest dieback,100,1000,2.0,3.5,6.0,4.604503,0.20,"Max impact for a total die back is 0.2C, while..."
AMOC,Atlantic meridional ocean circulation,50,500,1.4,4.0,8.0,2.656444,-0.50,NaN
ASI,Arctic winter sea ice,20,200,4.5,6.3,8.7,3.837086,0.60,NaN
ENSO,El Nino Southern Oscillation,100,1000,3.0,4.5,6.0,4.604503,0.00,From Armstrong McKay et al no estimate for max...
GIS,Greenland ice sheet,10000,100000,0.8,1.5,3.0,9.866793,0.13,NaN


# Generating interaction matrices

In [6]:
import scipy as sp
from scipy.stats import qmc
from numpy.random import Generator, PCG64

no_samples = 500
no_edges = 17
seed = 45643564852856852352111
sampler = qmc.LatinHypercube(d = no_edges,rng=Generator(PCG64(seed))) # 17 edges
sample = sampler.random(n=no_samples) # still in a [0,1]^17 hypercube 500 samples!
TEs = ['AMAZ', 'AMOC', 'ASI', 'ENSO', 'GIS', 'PERM', 'REEF', 'WAIS', 'WAM'] # see README.txt
TE_to_ind = {te:i for i,te in enumerate(TEs)}

folder = '../Data/'
df_interactions = pd.read_excel(folder+'Interactions_TableA2.xlsx',usecols='A:F',header=[0])
# df_interactions.head()

a = df_interactions['min'].to_numpy()
b = df_interactions['max'].to_numpy()
A = []
for r in range(no_samples):
    weights = a+sample[r,:]*(b-a)
    mat = np.zeros([len(TEs)]*2)
    for index, row in df_interactions.iterrows():
        i = TE_to_ind[row['Source']]
        j = TE_to_ind[row['Target']]
        mat[i,j] = weights[index]
    A.append(mat)
    # np.savetxt("../Data/Interaction matrices/A_%s.csv"%r,A[r])